In [ ]:
from functools import lru_cache
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
# import re
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer, util



%matplotlib inline

# Dataframe loading

In [5]:
df_prod = pd.read_csv('data/20250501-carrefour_prods.csv', parse_dates=['dateKey'])
df_loyalty = pd.read_csv('data/20250501-carrefour_loyalty.csv', parse_dates=['date'])
df_prod['totalTruePrice'] = df_prod['totalPrice'] + df_prod['totalImmediateDiscount']
display(df_prod)
display(df_loyalty)

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
0,2025-04-29,receipt,2,NaN,NaN,5X10 D FREEDENT ME,5X10 D FREEDENT ME,other,NaN,20.0,2,0.000,2.39,4.78,-1.43,3.35
1,2025-04-29,receipt,1,NaN,NaN,KG POULET FERMIER,KG POULET FERMIER,food,NaN,5.5,1,1.912,6.50,12.43,0.00,12.43
2,2025-04-29,receipt,1,NaN,NaN,750G OMEGA 3 DX SH,750G OMEGA 3 DX SH,other,NaN,20.0,1,0.000,5.39,5.39,0.00,5.39
3,2025-04-29,receipt,1,NaN,NaN,3 RECH RUBAN MAGI,3 RECH RUBAN MAGI,other,NaN,20.0,1,0.000,4.99,4.99,0.00,4.99
4,2025-04-29,receipt,1,NaN,NaN,132G BISC.COCO SSA,132G BISC.COCO SSA,food,NaN,5.5,1,0.000,2.45,2.45,0.00,2.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5014,2022-04-25,receipt,1,NaN,NaN,COURGETTE P10.5 CM,COURGETTE P10.5 CM,other,NaN,10.0,1,0.000,1.75,1.75,0.00,1.75
5015,2022-04-25,receipt,1,NaN,NaN,POMME ARIANE,POMME ARIANE,food,NaN,5.5,1,0.000,2.50,2.50,0.00,2.50
5016,2022-04-25,receipt,1,NaN,NaN,PIECE ANANAS EXTRA,PIECE ANANAS EXTRA,food,NaN,5.5,1,0.000,1.99,1.99,0.00,1.99
5017,2022-04-25,receipt,1,NaN,NaN,MENTHE FRAI,MENTHE FRAI,food,NaN,5.5,1,0.000,0.61,0.61,0.00,0.61


,operationId,date,earned,burned,itemLabel,promotionLabel,itemRd,loyaltyOperation
0,64070450536,2024-09-28,0.66,0.00,20 OEUFS DJP POULE AU SOL CRF,NaN,0.46,Paiement en caisse
1,64070450536,2024-09-28,0.66,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.20,Paiement en caisse
2,64063052302,2024-09-24,0.30,0.00,1KG COUSCOUS MOYEN CRF,NaN,0.30,Paiement en caisse
3,64072667140,2024-09-20,0.20,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.20,Paiement en ligne
4,64073135027,2024-09-13,0.00,-22.76,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
329,65151220627,2025-03-08,21.30,0.00,"ART RAYON FRUITS LEG TVA 5,5",NaN,1.29,Paiement en caisse
330,65151220627,2025-03-08,21.30,0.00,"ART RAYON POISSONNERIE TVA 5,5",NaN,12.91,Paiement en caisse
331,65151220627,2025-03-08,21.30,0.00,CHOU FLEUR PIECE,NaN,0.26,Paiement en caisse
332,65151220627,2025-03-08,21.30,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.25,Paiement en caisse


In [6]:
mapped_categories = {
    'Charcuterie' : 'food', 
    'Laits et Boissons végétales' : 'food',
    'Jus de fruits et légumes': 'food', 
    'Toasts et Pains de mie': 'food',
    'Yaourts et Fromages blancs': 'food',
    'Conserves et Bocaux': 'food',
    'Colas, Thés glacés, Sirops et Sodas': 'food',
    'Légumes': 'food',
    'Huiles, Vinaigres et Vinaigrettes': 'food',
    'Nettoyants vaisselle' : 'other',
    'Accessoires de ménage' : 'other',
    'Matériel de bureau' : 'other',
    'Fromages': 'food',
    'Epicerie salée': 'food',
    'Cheveux' : 'other',
    'Pains Burger, Sandwich et Wraps': 'food',
    'Eaux': 'food',
    'Viandes': 'food',
    'Lessives' : 'other',
    'Pizzas, Quiches et Tartes': 'food',
    'Apéritifs et Chips': 'food',
    'Fruits': 'food', 
    'Volaille et Rôtisserie': 'food',
    'Glaces et Sorbets': 'food',
    'Bio à Petit prix': 'food',
    'Gâteaux moelleux': 'food',
    'Apéritifs, Entrées et Snacking': 'food', 
    'Petit déjeuner': 'food',
    'Hygiène dentaire' : 'other', 
    'Cave à Vins': 'food',
    'Boucherie': 'food',
    'Poissons et Fruits de mer': 'food',
    'Œufs': 'food',
    'Poissonnerie': 'food',
    'Essuie-tout, Papier toilette et Mouchoirs' : 'other',
    'Confiseries et Chocolats': 'food', 
    'RETURNABLE_BAG' : 'other', 
    'Hygiène intime ' : 'other',
    'Désodorisants et Bougies' : 'other',
    'Toutes nos régions': 'food',
    'Produits nettoyants' : 'other', 
    'Riz, Purées et Féculents' : 'food',
    'Ingrédients pour cuisiner' : 'food',
    'Sauces froides' : 'food',
    'Pains frais' : 'food',
    'Bières et Cidres' : 'food',
    'Repas de Pâques' : 'food',
    'Le Marché' : 'food',
    'Beurres et Crèmes' : 'food',
    'Premiers soins et Préservatifs' : 'other',
    'Nintendo Switch' : 'other',
    'Sucres, Farines et Aide à la pâtisserie' : 'food',
    'Viennoiseries et Brioches fraîches' : 'food', 
    'Corps' : 'other'
}

In [7]:
df_prod.loc[df_prod.category.isna(), "category"] = df_prod.loc[df_prod.category.isna(), "subCategory"].map(mapped_categories)

# Group analysis

The method unstack allows to view the `groupby` results in a natural table. This is actually a pivot on the column by which one applies `groupby`. 
The `reset_index()` method actually returns it as a table with index as rows instead (no pivoting). One or another is useful.

In [9]:
df_loyalty[['itemLabel', 'date', 'loyaltyOperation', 'itemRd']].groupby(by=[df_loyalty.date.dt.year, 'itemLabel'], sort=True).sum(numeric_only=True).sort_values(by=['date', 'itemRd'], ascending = [False, False]).unstack(fill_value=0)

itemRd                                 \
itemLabel 100G GRANA PADANO RAPE AOP CRF 125G MOZZARELLA BUFALA BIO CRF   
date                                                                      
2024                                0.00                           0.00   
2025                                0.58                           0.77   

                                                             \
itemLabel 125G MOZZARELLA VACHE 12X125G YAOURTS PAT LA LAIT   
date                                                          
2024                       1.19                        0.00   
2025                       0.00                        1.34   

                                                                        \
itemLabel 16X125G BIF NATURE ACTIVIA OD 16X125G YRT FRUIT 0% CRF CLASS   
date                                                                     
2024                               1.35                           0.61   
2025                               0.00                           0.00   

                                                                 \
itemLabel 1KG COUSCOUS MOYEN CRF 1KG FARINE BLE FLUID T45 CRF C   
date                                                              
2024                         0.3                           0.13   
2025                         0.0                           0.00   

                                                               ...  \
itemLabel 1KG FARINE BLE T45 CRF CLASSIC 1KG FILET POULET PLK  ...   
date                                                           ...   
2024                                0.13                 0.00  ...   
2025                                0.00                 1.99  ...   

                                                           \
itemLabel RICORE RECHARGE 290G RUBAN TRANSPT 550 19MMX33M   
date                                                        
2024                      0.00                       1.93   
2025                      3.24                       0.00   

                                                       \
itemLabel SALADE BATAVIA PIECE ST 1,5KG POMME GALA FR   
date                                                    
2024                      0.37                   0.64   
2025                      0.00                   0.00   

                                                                               \
itemLabel ST 200G HARICOT.MUNGO 0,99 ST 2PCES MAIS DOUX HF ST 400G EPINARD FR   
date                                                                            
2024                            0.15                   0.0                0.0   
2025                            0.00                   0.9                0.5   

                                                                    \
itemLabel ST 500G BETTERAV PRIM FQC AGRO ST 500G BETTERAVE FQC AGR   
date                                                                 
2024                                0.27                      0.25   
2025                                0.00                      0.00   

                                          
itemLabel SWITCH SET MARIO KART 8 PASS C  
date                                      
2024                                10.5  
2025                                 0.0  

[2 rows x 102 columns]

In [ ]:
grouped_loyalty_year = df_loyalty[['itemLabel', 'date', 'loyaltyOperation', 'itemRd']].groupby(by=[df_loyalty.date.dt.year, 'itemLabel'], sort=True).sum(numeric_only=True).sort_values(by=['date', 'itemRd'], ascending = [False, False]).reset_index()
display(grouped_loyalty_year.head(10))

In [ ]:
display(grouped_loyalty_year[grouped_loyalty_year.date == 2024].nlargest(10, columns='itemRd'))

,date,itemLabel,itemRd
58,2024,"ART RAYON POISSONNERIE TVA 5,5",15.81
59,2024,SWITCH SET MARIO KART 8 PASS C,10.50
60,2024,"ART RAYON FRUITS LEG TVA 5,5",9.84
61,2024,BANANE CRF BIO MH 5 FRUITS,9.09
62,2024,2X6TR 2X215G SF NOR,6.60
63,2024,2X75ML DENT PSA NETT INTENS OB,4.70
64,2024,2X75ML DENT S&G CALM ORIGIN OB,4.25
65,2024,2X75ML PSA GENC&EMAIL ORIG OB,4.24
66,2024,1X8L FF BP MATIN LEGER ECREME,4.08
67,2024,3X75ML DENT WN TP SENS SIGNAL,4.07


In [103]:
# Group by 'category' and sum the 'amount' column
spending_by_category = df_prod.groupby(['category', df_prod.dateKey.dt.year]).sum(numeric_only=True)

print("\nTotal Spending by Category:")
display(spending_by_category)


Total Spending by Category:


countVisits           ean       cdbase  vatPercentage  \
category dateKey                                                          
food     2022             947  0.000000e+00          0.0         4482.5   
         2023            1620  0.000000e+00          0.0         7386.5   
         2024            1745  6.645180e+14  781106656.0         7293.0   
         2025             533  7.403638e+13  100131650.0         2370.5   
other    2022             340  0.000000e+00          0.0         4660.0   
         2023             476  0.000000e+00          0.0         6010.0   
         2024             465  3.411904e+14  313664675.0         4640.0   
         2025              88  6.492628e+13   55207784.0          960.0   

                  totalQuantity  totalWeight    unitPrice  totalPrice  \
category dateKey                                                        
food     2022              1163       91.723  2559.725000     3046.18   
         2023              1938      192.193  4475.635333     5454.36   
         2024              2064      239.980  4858.790000     5683.34   
         2025               614       98.519  1603.980000     1904.38   
other    2022               331       -5.721  1519.430000     1763.48   
         2023               474        0.000  2543.440000     3165.19   
         2024               551       -1.200  1937.386667     2816.82   
         2025               103        0.000   352.610000      609.50   

                  totalImmediateDiscount  totalTruePrice  
category dateKey                                          
food     2022                    -133.34         2912.84  
         2023                    -330.49         5123.87  
         2024                    -165.56         5517.78  
         2025                     -63.04         1841.34  
other    2022                    -165.00         1598.48  
         2023                    -382.10         2783.09  
         2024                    -272.27         2544.55  
         2025                     -41.68          567.82

In [ ]:
df_prod['yearMonth'] = df_prod.dateKey.dt.to_period('M')
# Group by year_month and product_id, then sum the quantities
grouped_month = df_prod.groupby(['yearMonth', 'productLabel']).agg(
    {'totalQuantity' : 'sum', 'totalWeight': 'sum', 'unitPrice' : 'mean', 'totalPrice' : 'sum', 'totalImmediateDiscount' : 'sum', 'totalTruePrice' : 'sum'}
).reset_index()

# Sort by year_month and quantity in descending order
sorted_grouped_month_quant = grouped_month.sort_values(by=['yearMonth', 'totalQuantity'], ascending=[True, False])
sorted_grouped_month_weight = grouped_month.sort_values(by=['yearMonth', 'totalWeight'], ascending=[True, False])

# Get the top 10 items for each month
top5_quant_by_month = sorted_grouped_month_quant.groupby('yearMonth').head(5)  # Top N records per group
top5_weight_by_month = sorted_grouped_month_weight.groupby('yearMonth').head(5)  # Top N records per group
display(top5_quant_by_month[top5_quant_by_month.yearMonth.dt.year == 2024])
display(top5_weight_by_month[top5_weight_by_month.yearMonth.dt.year == 2024])

,yearMonth,productLabel,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
2444,2024-01,KIWI JAUNE PIECE,20,0.000,0.723333,14.20,-2.32,11.88
2446,2024-01,KIWI VERT PIECE,12,0.000,0.640000,7.48,-1.88,5.60
2382,2024-01,24 PS EXTRA LONG C,9,0.000,1.550000,13.95,-1.24,12.71
2405,2024-01,6X1.5L CRISTALINE,9,0.000,1.140000,10.26,0.00,10.26
2445,2024-01,KIWI VERT FQC,8,0.000,0.690000,5.52,-1.52,4.00
2582,2024-02,KIWI VERT PIECE,17,0.000,0.590000,10.03,-1.74,8.29
2601,2024-02,Nettoyant Optique Dégraissant Anti Trace VU,8,0.000,2.790000,22.32,0.00,22.32
2536,2024-02,6X1.5L CRISTALINE,7,0.000,1.140000,7.98,0.00,7.98
2597,2024-02,Mouchoirs Confort CARREFOUR SOFT,6,0.000,3.790000,22.74,3.42,26.16
2568,2024-02,CITRON VERT PIECE,5,0.000,0.500000,2.50,-0.50,2.00


,yearMonth,productLabel,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
2453,2024-01,PDT CONSOMMATION,5,4.487,1.898000,10.46,0.0,10.46
2477,2024-01,TOMATE RDE CHARNUE,2,2.010,2.490000,5.27,0.0,5.27
2467,2024-01,POMME PINK LADY KG,2,1.744,1.990000,3.47,0.0,3.47
2451,2024-01,PATATE DOUCE VRAC,1,1.724,1.990000,3.43,0.0,3.43
2466,2024-01,POMME ARIANE VRAC,1,1.304,2.990000,3.90,0.0,3.90
2611,2024-02,PDT CONSOMMATION,2,2.854,1.990000,5.68,0.0,5.68
2590,2024-02,MANDARINE CRF,2,1.959,1.990000,3.90,0.0,3.90
2626,2024-02,POMME GRANNY GROSS,2,1.572,2.990000,4.70,0.0,4.70
2562,2024-02,CAROTTE VRAC,1,1.036,2.490000,2.58,0.0,2.58
2627,2024-02,POMME PINK LADY KG,1,0.915,1.990000,1.82,0.0,1.82


In [ ]:
# Group by year_month and product_id, then sum the quantities
grouped_year = df_prod.groupby(['year', 'productLabel']).agg(
    {'totalQuantity' : 'sum', 'totalWeight': 'sum', 'unitPrice' : 'mean', 'price' : 'sum', 'totalImmediateDiscount' : 'sum'}).reset_index()

# Sort by year_month and quantity in descending order
sorted_grouped_year_quant = grouped_year.sort_values(by=['year', 'totalQuantity'], ascending=[True, False])
sorted_grouped_year_weight = grouped_year.sort_values(by=['year', 'totalWeight'], ascending=[True, False])

# Get the top 10 items for each month
top10_quant_by_year = sorted_grouped_year_quant.groupby('year').head(10)  # Top N records per group
top10_weight_by_year = sorted_grouped_year_weight.groupby('year').head(10)  # Top N records per group
display(top10_quant_by_year)
display(top10_weight_by_year)

,year,name,totalQuantity,totalWeight,unitPrice,price,immediateDiscount
447,2022,KIWI PIECE,129,0.000,0.564000,4.893333,-0.390000
51,2022,16X125 VELOUTE NAT,19,0.000,3.778750,3.778750,0.000000
356,2022,CHAUSSON AUX POMME,18,0.000,0.750000,1.934000,0.000000
291,2022,AVOCAT PIECE,16,0.000,1.170000,3.888000,-0.266000
372,2022,CONCOMBRE PIECE,16,0.000,1.111250,2.222500,-0.022500
614,2022,VIENNOISERIE AUX A,16,0.000,0.755000,3.020000,0.000000
467,2022,MANGUE PIECE,14,0.000,1.484000,3.022000,-0.324000
290,2022,AUBERGINE VIOLETTE,13,6.118,2.860769,1.358462,0.000000
336,2022,BT 3X110MOUCH.CONF,13,0.000,3.475714,3.475714,-0.158214
485,2022,NESCAFE SPEC.FILTR,13,0.000,6.120833,6.120833,-0.050000


,year,name,totalQuantity,totalWeight,unitPrice,price,immediateDiscount
603,2022,TOMATE GRAPPE FRAN,8,10.050,2.365000,2.932500,0.0
290,2022,AUBERGINE VIOLETTE,13,6.118,2.860769,1.358462,0.0
544,2022,POMME GRANNY GROSS,7,5.184,2.442857,1.822857,0.0
512,2022,PDT CONSERVATION,4,4.685,1.690000,2.020000,0.0
548,2022,PORC SAUTE EPAULE,5,4.422,6.120000,4.865000,0.0
546,2022,POMME PINK LADY,4,4.348,3.590000,3.937500,0.0
376,2022,COURGETTE,5,4.223,2.650000,2.176000,0.0
299,2022,BANANE OPEN TOP VR,3,3.539,1.690000,1.993333,0.0
596,2022,TARO VRAC,4,3.513,6.640000,5.902500,0.0
543,2022,POMME GALA MOYENNE,3,3.293,2.390000,2.827500,0.0


# Partial string matching

Thanks to the grouping analysis, we get a grasp of the top products on which we want to make further analysis like actual price evolutions, purchased quantities, variations of item types, etc.

We use regular expressions as well `str.contains()` and fuzzy matching thanks to `rapidfuzz` (for scalability sake).

For each date we have a matching products. In order to perform good matching, we should need to preprocess the labels as to have the dates as tokens that helps to do the matching.

Use sentence transformers to do cosine similarity embedding, and then for uncertain ones (similarity scores < 0.5), do some normal fuzzy ratio stuff to get the correct matching label.

In [28]:
df_prod.query("productLabel.str.contains('tomate', case=False) & totalWeight > 0").value_counts(['recordType', 'productLabel', 'unitPrice'])

recordType  productLabel        unitPrice
receipt     KG TOMATE GRAP FR   2.99         17
                                1.99          8
            TOMATE GRAPPE FRAN  2.99          5
                                1.99          4
            KG TOMATE GRAP FR   3.99          3
            TOMATE GRAPPE FRAN  2.49          3
            KG TOMATE GRAP FR   2.49          3
                                2.79          2
            TOMATE GRAPPE FRAN  2.29          2
                                1.39          2
            TOMATE GRAPPE       2.99          2
            TOMATE GRAPPE FRAN  1.59          2
            TOMATE GRAPPE       3.29          2
            KG TOMATE GRAP FR   3.89          2
                                2.29          2
            TOMATE ALLONGEE     2.99          2
            TOMATE GRAPPE VRAC  2.99          2
            TOMATE RONDE PET    3.69          2
            TOMATE GRAPPE IMPO  1.99          2
            KG TOMATE GRAP FR   1.29          

In [55]:
def preprocess(text: str): 
    text = text.lower()
    # Normalize to decomposed form (split base characters and diacritics)
    nfkd_form = unicodedata.normalize('NFKD', text)
    text = u"".join([c for c in nfkd_form if not unicodedata.combining(c)])
    
    # Use regex to remove diacritics (Mn = Mark, Nonspacing)
    # text = re.sub(r'\p{Mn}', '', text, flags=re.UNICODE)
    # text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return text.strip()

def fuzzy_match(row, choices, scorer=fuzz.WRatio, processor=None, threshold=None):
    """
    Perform fuzzy matching for a single row against a list of choices based on date criterion.
    Returns the best match and its score if above the threshold.
    """
    result = process.extractOne(row, choices, scorer=scorer, processor=processor, score_cutoff=threshold)
    return result[0] if result is not None else None


In [ ]:
def load_bert_model():
    return SentenceTransformer("all-MiniLM-L6-v2")


def embedding_to_df(model, df: pd.DataFrame, column_name: str, new_column_name: str = "embeddingLabel"):
    """In-place computation of embedding"""
    df.loc[df_loyalty[column_name].notna(), new_column_name] = df_loyalty.loc[
        df[column_name].notna(), column_name
    ].apply(lambda x: cached_encode(model, x))


# Define a cached version of the embedding function
@lru_cache(maxsize=None)  # Cache all results
def cached_encode(model, text):
    return model.encode(preprocess(text))


def match_products(extracted_entities, known_products, model):
    # Encode both lists of texts
    extracted_embeddings = model.encode(extracted_entities, convert_to_tensor=True)
    known_embeddings = model.encode(known_products, convert_to_tensor=True)

    # Compute cosine similarity
    similarity_scores = util.cos_sim(extracted_embeddings, known_embeddings)

    # Find the best match for each extracted entity
    matched_products = []
    for i, entity in enumerate(extracted_entities):
        best_match_idx = similarity_scores[i].argmax().item()
        matched_products.append(
            (
                entity,
                known_products[best_match_idx],
                similarity_scores[i][best_match_idx].item(),
            )
        )

    return matched_products

In [ ]:


# Load a pre-trained Sentence Transformer model
model = load_bert_model()

In [ ]:

embedding_to_df(model, df_loyalty, 'itemLabel')
embedding_to_df(model, df_prod, 'productLabel')


# # Apply the cached function to generate embeddings
# df_loyalty.loc[df_loyalty['itemLabel'].notna(), 'embeddingLabel'] = df_loyalty.loc[
#     df_loyalty['itemLabel'].notna(), 'itemLabel'
# ].apply(lambda x: cached_encode(x))

# df_prod['embeddingLabel'] = None
# df_prod.loc[df_prod.dateKey.isin(df_loyalty.date.unique()), 'embeddingLabel'] = df_prod.loc[
#     df_prod.dateKey.isin(df_loyalty.date.unique()), 'productLabel'
# ].apply(lambda x: cached_encode(x))

In [ ]:
# # Generate embeddings for product names in both dataframes
# df_loyalty.loc[df_loyalty['itemLabel'].notna(), 'embeddingLabel'] = df_loyalty.loc[df_loyalty['itemLabel'].notna(), 'itemLabel'].apply(lambda x: model.encode(preprocess(x)))
# df_prod['embeddingLabel'] = None
# df_prod.loc[df_prod.dateKey.isin(df_loyalty.date.unique()) ,'embeddingLabel'] = df_prod.loc[df_prod.dateKey.isin(df_loyalty.date.unique()) ,'productLabel'].apply(lambda x: model.encode(preprocess(x)))

In [60]:
df_prod['embeddingLabel'].notna().sum()

np.int64(1637)

In [61]:
# Group rows by date
grouped_df_loyalty = df_loyalty[df_loyalty.embeddingLabel.notna()].groupby('date')
grouped_df_prod = df_prod[df_prod.embeddingLabel.notna()].groupby('dateKey')

In [ ]:


# Create a list to store matches
matches = []

# Iterate over unique dates in df1
for date, group1 in grouped_df_loyalty:
    # Check if the same date exists in df2
    # Drop rows with NaN values
    group1 = group1.reset_index(drop=True)

    if date in grouped_df_prod.groups:
        group2 = grouped_df_prod.get_group(date)

        group2 = group2.reset_index(drop=True)
        
        # Compute pairwise cosine similarity between embeddings
        similarity_matrix = cosine_similarity(np.vstack(group1['embeddingLabel']), np.vstack(group2['embeddingLabel']))
        # print(similarity_matrix)
        
        # Find the best match for each row in group1
        for i, row1 in group1.iterrows():
            # print("i", i)
            # best_match_idx = np.argmax(similarity_matrix[i])

            sort_indices = np.argsort(similarity_matrix[i])[::-1]
            best_match = group2.iloc[sort_indices[0]]
            if len(sort_indices) > 1:
                second_best_match = group2.iloc[sort_indices[1]]
                matches.append({
                    "df_loyalty_label": row1['itemLabel'],
                    "df_prod_label": best_match['productLabel'],
                    "df_prod_label2" : second_best_match['productLabel'],
                    "similarity_score": similarity_matrix[i][sort_indices[0]],
                    "similarity_score2" : similarity_matrix[i][sort_indices[1]],
                    "date": date
                })
            else:
                matches.append({
                    "df_loyalty_label": row1['itemLabel'],
                    "df_prod_label": best_match['productLabel'],
                    "similarity_score": similarity_matrix[i][sort_indices[0]],
                    "date": date
                })
            # use argsort
            
            
            

# Convert matches to a dataframe
matches_df = pd.DataFrame(matches)

In [63]:
matches_df.query("(~df_loyalty_label.str.contains('ART RAYON')) & (similarity_score < 0.5)")

,df_loyalty_label,df_prod_label,df_prod_label2,similarity_score,similarity_score2,date
9,1X8L FF BP MATIN LEGER ECREME,4X115G RIZ VAN LA,20 OEUFS DJP PLE A,0.461484,0.455263,2024-05-24
22,20 OEUFS DJP POULE AU SOL CRF,Œufs frais poules au sol CARREFOUR CLASSIC',Eau gazeuse minérale naturelle ST-YORRE,0.488403,0.397678,2024-06-27
61,20 OEUFS DJP POULE AU SOL CRF,Œufs frais poules au sol CARREFOUR CLASSIC',Blanc de dinde fumé FLEURY MICHON,0.488403,0.420442,2024-08-26
89,20 OEUFS DJP POULE AU SOL CRF,Œufs frais poules au sol CARREFOUR CLASSIC',Papier toilette Ultra Confort CARREFOUR ESSENTIAL,0.488403,0.332010,2024-10-07
117,BQ250G TT CERISE ALL SIMPL HF,2X250G PL BEUR GAS,24P KIRI 432G,0.435329,0.402839,2024-12-20
130,BQ250G TT CERISE ALL SIMPL HF,130G BIO LE SABLE,BQ TOM CERISE ALL,0.427958,0.394036,2025-01-14
139,BQ250G TT CERISE ALL SIMPL HF,PDT 2KG FPP FQC,420GPIZZA FDB CHOR,0.426157,0.397333,2025-01-21
194,MENTHE FRAICHE BOTTE,PIMENT VERT,MENTHE,0.447594,0.424051,2025-03-22
201,BQ250G TT CERISE ALL SIMPL HF,BQ TOM CERISE ALL,FILET SAUMON FQC A,0.394036,0.367668,2025-03-29
214,BQ250G TT CERISE ALL SIMPL HF,75BDX RG MALESANML,BV SEL FIN IODE,0.425396,0.405823,2025-04-05


In [64]:
matches_df.query("(~df_loyalty_label.str.contains('ART RAYON')) & (similarity_score > 0.5) & (similarity_score < 0.7)").head(50)

,df_loyalty_label,df_prod_label,df_prod_label2,similarity_score,similarity_score2,date
4,ST 500G BETTERAVE FQC AGR,BETTERAVE FQC,500G SH41 SS HDP,0.641942,0.475685,2024-05-17
10,1L BOISSON SOJA NATUR CARF BIO,Boisson végétale au soja nature Bio CARREFOUR BIO,Lait Sans lactose UHT Ecrémé Matin Léger LACTEL,0.671037,0.452734,2024-05-24
15,4X138G CROUSTILLE EMTL. BELIN,CROUSTILLE EMTL,125GX8 PAN0% JNES,0.667249,0.519309,2024-06-07
23,1L BOISSON SOJA NATUR CARF BIO,Boisson végétale au soja nature Bio CARREFOUR BIO,Sucre roux pure canne blond BLONVILLIERS,0.671037,0.426967,2024-06-27
25,400 ML LT CORPS NUTRI ARG LPM,LPM LAIT ARG400 GE,75ML GOMMAGE VISAG,0.569146,0.464015,2024-07-06
33,FLT 1 KG PDT CONSO 0.99 SIMPL,FLT 1KG P.DE TERRE,500G OIGNON RGE FR,0.566642,0.465731,2024-07-26
34,BQ250G TT CERISE ALL SIMPL HF,BQ 200G CHAMP FQC,ST 500G BETTERAVE,0.571576,0.396322,2024-07-26
48,RUBAN TRANSPT 550 19MMX33M,Ruban adhésif 33 x 19mm SCOTCH,Liquide Vaisselle Vinaigre & Citron Bi-Activ' ...,0.552965,0.226535,2024-08-09
49,BANANE CRF BIO MH 5 FRUITS,Bananes Bio CARREFOUR BIO,Légumes pour couscous Bio JARDIN BIO ETIC,0.617262,0.419397,2024-08-09
51,1L BOISSON SOJA NATUR CARF BIO,Boisson végétale au soja nature Bio CARREFOUR...,Légumes pour couscous Bio JARDIN BIO ETIC,0.671037,0.475065,2024-08-09


In [48]:
df_prod.query("(dateKey == @pd.to_datetime('2024-05-24'))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice,embeddingLabel
1552,2024-05-24,receipt,1,NaN,NaN,PASTEIS DE NATA X1,PASTEIS DE NATA X1,food,NaN,5.5,4,0.000,0.65,2.60,0.00,2.60,"[-0.041956294, 0.024259489, -0.03509082, -0.04..."
1553,2024-05-24,order,1,7.613036e+12,6397521.0,Eau minérale naturelle VITTEL,eau-minerale-naturelle-vittel,food,Eaux,NaN,3,0.000,2.94,8.82,2.94,11.76,"[-0.07479005, 0.034277786, 0.005259288, 0.0280..."
1554,2024-05-24,receipt,1,NaN,NaN,TORSADE CHOCO X1,TORSADE CHOCO X1,food,NaN,5.5,2,0.000,0.79,1.58,0.00,1.58,"[-0.080259725, -0.012456403, -0.061640214, -0...."
1555,2024-05-24,receipt,1,NaN,NaN,275G PAT CARAMEL S,275G PAT CARAMEL S,food,NaN,5.5,1,0.000,3.44,3.44,0.00,3.44,"[-0.030887973, 0.05882252, -0.027603066, 0.094..."
1556,2024-05-24,order,1,3.560070e+12,3635056.0,Nectar de goyave CARREFOUR SELECTION,nectar-de-goyave-carrefour-selection,food,Jus de fruits et légumes,NaN,0,0.000,2.25,0.00,0.00,0.00,"[0.06334139, -0.019752303, -0.012780657, 0.003..."
1557,2024-05-24,receipt,1,NaN,NaN,4X115G RIZ NAT LA,4X115G RIZ NAT LA,food,NaN,5.5,1,0.000,1.69,1.69,0.00,1.69,"[0.0048119808, 0.07058744, -0.006415787, -0.04..."
1558,2024-05-24,receipt,1,NaN,NaN,4X115G SEMOULE VAN,4X115G SEMOULE VAN,food,NaN,5.5,1,0.000,1.69,1.69,-1.69,0.00,"[0.04674312, 0.08978423, -0.024009941, -0.0634..."
1559,2024-05-24,receipt,1,NaN,NaN,CAROTTE FQC VRAC,CAROTTE FQC VRAC,food,NaN,5.5,1,0.593,1.99,1.18,0.00,1.18,"[-0.08201648, 0.01344691, -0.09474176, 0.02093..."
1560,2024-05-24,receipt,1,NaN,NaN,CROQUE EMMENTAL,CROQUE EMMENTAL,food,NaN,5.5,1,0.000,1.85,1.85,0.00,1.85,"[0.009451581, 0.07780721, 0.013974983, 0.00159..."
1561,2024-05-24,receipt,1,NaN,NaN,3X125G MOZZARELLA,3X125G MOZZARELLA,food,NaN,5.5,1,0.000,3.58,3.58,0.00,3.58,"[-0.04300408, -0.014758907, -0.05724107, 0.032..."


In [66]:
fuzz.token_sort_ratio(preprocess(item1), preprocess(item2))

35.64356435643564

In [54]:
preprocess(item1)

'banane crf bio mh 5 fruits'

In [55]:
preprocess(item2)

'bananes bio carrefour bio'

In [111]:
df_loyalty['matchedProductLabel'] = None

In [54]:
# Iterate over unique dateKeys in df1
for dateKey in df_loyalty['date'].unique():
    # Filter both DataFrames by the current dateKey
    # df1_subset = df_prod[df_prod['dateKey'] == dateKey]
    # df2_subset = df_loyalty[df_loyalty['date'] == dateKey]
    # print(df2_subset)

    # Extract unique product labels from df2_subset as choices
    choices = df_prod.loc[df_prod.dateKey == dateKey, 'productLabel'].unique()
    # print(choices)

    # Apply fuzzy matching to product labels in df1_subset
    result = df_loyalty.loc[df_loyalty.date == dateKey, 'itemLabel'].apply(
        fuzzy_match, 
        choices=choices,
        scorer=fuzz.partial_token_sort_ratio,
        processor = preprocess
    )
    # if result is None:
    #     result = df_prod.loc[df_prod['dateKey'] == dateKey, 'productLabel'].apply(
    #     fuzzy_match, 
    #     choices=choices,
    #     scorer=fuzz.token_sort_ratio, 
    #     processor = preprocess
    #     )
    #     print(result)
    df_loyalty.loc[df_loyalty.date == dateKey, 'matchedProductLabel'] = result

NameError: name 'fuzzy_match' is not defined

In [117]:
# Combine all subsets into a single DataFrame
df_loyalty.loc[df_loyalty.matchedProductLabel.isna(), ['date', 'itemLabel', 'matchedProductLabel']]

,date,itemLabel,matchedProductLabel
4,2024-09-13,NaN,None
18,2024-09-11,NaN,None
21,2024-09-11,NaN,None
22,2024-09-10,NaN,None
23,2024-09-07,NaN,None
...,...,...,...
297,2024-07-02,NaN,None
309,2025-03-22,NaN,None
319,2025-03-15,NaN,None
325,2025-03-08,NaN,None


In [118]:
df_loyalty.loc[(df_loyalty.matchedProductLabel.isna()) & ((df_loyalty.itemLabel.notna())),  ['date', 'itemLabel', 'itemRd']]

,date,itemLabel,itemRd


In [119]:
df_loyalty[df_loyalty.matchedProductLabel.notna()].tail(20)

,operationId,date,earned,burned,itemLabel,promotionLabel,itemRd,loyaltyOperation,matchedProductLabel
311,65151304770,2025-03-22,4.53,0.0,"ART RAYON FRUITS LEG TVA 5,5",NaN,1.16,Paiement en caisse,PDT VAP BLANCHE 2K
312,65151304770,2025-03-22,4.53,0.0,BATAVIA PIECE,NaN,0.22,Paiement en caisse,BATAVIA PIECE
313,65151304770,2025-03-22,4.53,0.0,FLT 2KG PDT BLC VAP CRF FR,NaN,0.45,Paiement en caisse,PDT VAP BLANCHE 2K
314,65151304770,2025-03-22,4.53,0.0,100G GRANA PADANO RAPE AOP CRF,NaN,0.58,Paiement en caisse,100G GRANA PADANO
315,65151304770,2025-03-22,4.53,0.0,MENTHE FRAICHE BOTTE,NaN,0.16,Paiement en caisse,MENTHE
316,65151304770,2025-03-22,4.53,0.0,FLT 2KG ORANGE DESSERT CRF HF,NaN,0.52,Paiement en caisse,ORANGE DESSERT 2KG
317,65151304770,2025-03-22,4.53,0.0,BANANE CRF BIO MH 5 FRUITS,NaN,0.29,Paiement en caisse,BANANE CRF BIO
318,65151304770,2025-03-22,4.53,0.0,PERSIL PLAT FRAIS BOTTE MAX,NaN,0.38,Paiement en caisse,PERSIL PLAT
320,65146373916,2025-03-15,10.62,0.0,"ART RAYON BOUCHERIE TVA 5,5",NaN,6.60,Paiement en caisse,PDT VAP BLANCHE 2K
321,65146373916,2025-03-15,10.62,0.0,BATAVIA PIECE,NaN,0.23,Paiement en caisse,BATAVIA PIECE


In [91]:
df_loyalty[df_loyalty.matchedProductLabel.notna()].query('itemLabel.str.contains("ARIEL")')

,operationId,date,earned,burned,itemLabel,promotionLabel,itemRd,loyaltyOperation,matchedProductLabel
184,65143918436,2025-01-14,66.51,0.0,ARIEL LIQ 3X22D ACTIVE,NaN,9.94,Paiement en caisse,FD ARIEL LIQ 3X22D
186,65143918436,2025-01-14,66.51,0.0,LTX3 ARIEL LIQ 22D UNSTOPPABLE,NaN,19.90,Paiement en caisse,FD ARIEL LIQ 3X22D
197,65152430725,2025-01-05,42.84,0.0,ARIEL LIQUIDE 34D ORIGINAL,NaN,3.57,Paiement en ligne,Lessive Liquide Original ARIEL
198,65152430725,2025-01-05,42.84,0.0,ARIEL LIQUIDE 31D PROTECTION,NaN,14.28,Paiement en ligne,Lessive Liquide Extra + Color Protection ARIEL
200,65152430725,2025-01-05,42.84,0.0,ARIEL LIQUIDE 31D ULTRA,NaN,3.57,Paiement en ligne,Lessive Liquide Ultra Détachant ARIEL


In [14]:
df_prod.query("(dateKey == @pd.to_datetime('2025-01-05'))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
499,2025-01-05,order,1,3.560070e+12,3164301.0,Papier toilette Ultra Confort CARREFOUR ESSENTIAL,papier-toilette-ultra-confort-carrefour-essential,other,"Essuie-tout, Papier toilette et Mouchoirs",NaN,4,0.000,4.59,18.36,2.76,21.12
500,2025-01-05,order,1,3.560071e+12,6557620.0,Œufs frais poules au sol CARREFOUR CLASSIC',oeufs-frais-poules-au-sol-carrefour-classic,food,Œufs,NaN,2,0.000,4.55,9.10,1.37,10.47
501,2025-01-05,order,1,3.266980e+12,107794.0,Poulet jaune PAC Label Rouge FERMIERS DE LOUE,poulet-jaune-pac-label-rouge-fermiers-de-loue,food,Volaille et Rôtisserie,NaN,1,1.482,8.52,8.52,0.00,8.52
502,2025-01-05,order,1,3.000001e+12,434072.0,Avocat,avocat,food,Légumes,NaN,2,0.000,1.39,2.78,0.00,2.78
503,2025-01-05,order,1,3.033490e+12,6287309.0,Yaourt nature DANONE,yaourt-nature-danone,food,Yaourts et Fromages blancs,NaN,1,0.000,3.35,3.35,0.00,3.35
504,2025-01-05,order,1,8.700216e+12,7739158.0,Lessive Liquide Original ARIEL,lessive-liquide-original-ariel,other,Lessives,NaN,1,0.000,11.89,11.89,0.00,11.89
505,2025-01-05,order,1,9.713236e+12,7537445.0,Sacs réutilisables consignés,sacs-reutilisables-consignes,other,RETURNABLE_BAG,NaN,2,0.000,0.35,0.70,0.00,0.70
506,2025-01-05,order,1,3.270190e+12,1043446.0,Eau de source CARREFOUR CLASSIC',eau-de-source-carrefour-classic,food,Eaux,NaN,2,0.000,1.62,3.24,0.32,3.56
507,2025-01-05,order,1,8.700216e+12,7746767.0,Lessive Liquide Ultra Détachant ARIEL,lessive-liquide-ultra-detachant-ariel,other,Lessives,NaN,1,0.000,11.89,11.89,0.00,11.89
508,2025-01-05,order,1,3.276559e+12,7910387.0,Tomates cerises allongées CARREFOUR SIMPL,tomates-cerises-allongees-carrefour-simpl,food,Légumes,NaN,2,0.000,0.99,1.98,0.00,1.98


In [11]:
df_prod.query("(dateKey == @pd.to_datetime('2024-09-20'))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
959,2024-09-20,receipt,1,NaN,NaN,3X70G EMMENTAL RAP,3X70G EMMENTAL RAP,food,NaN,5.5,1,0.000,1.99,1.99,0.00,1.99
960,2024-09-20,receipt,1,NaN,NaN,200G MIMOLETTE TRA,200G MIMOLETTE TRA,food,NaN,5.5,1,0.000,1.95,1.95,-0.39,1.56
961,2024-09-20,order,2,3.560070e+12,3731614.0,Filet de poulet,filet-de-poulet,food,Volaille et Rôtisserie,NaN,0,0.000,7.79,0.00,0.00,0.00
962,2024-09-20,receipt,1,NaN,NaN,FLT 1.5KG ORANGE,FLT 1.5KG ORANGE,food,NaN,5.5,1,0.000,3.99,3.99,0.00,3.99
963,2024-09-20,order,1,8.718952e+12,7710113.0,Liquide Vaisselle Vinaigre et Menthol Bi-Activ...,liquide-vaisselle-vinaigre-et-menthol-bi-activ...,other,Nettoyants vaisselle,NaN,2,0.000,4.99,9.98,3.00,12.98
964,2024-09-20,receipt,1,NaN,NaN,CREME FORESTIERE,CREME FORESTIERE,food,NaN,5.5,1,0.000,2.79,2.79,0.00,2.79
965,2024-09-20,receipt,1,NaN,NaN,800G BRANDADE MORU,800G BRANDADE MORU,food,NaN,5.5,1,0.000,9.50,9.50,0.00,9.50
966,2024-09-20,receipt,1,NaN,NaN,CAROTTE FQC VRAC,CAROTTE FQC VRAC,food,NaN,5.5,1,0.619,1.39,0.86,0.00,0.86
967,2024-09-20,order,1,9.713236e+12,7537445.0,Sacs réutilisables consignés,sacs-reutilisables-consignes,other,RETURNABLE_BAG,NaN,0,0.000,0.35,0.00,0.00,0.00
968,2024-09-20,receipt,1,NaN,NaN,6X1.5L CRISTALINE,6X1.5L CRISTALINE,food,NaN,5.5,1,0.000,1.14,1.14,0.00,1.14


In [141]:
df_prod.query("(dateKey == @pd.to_datetime('2024-08-09')) & (productLabel.str.contains('RUBAN', case=False))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice,matchedProductLabel
1224,2024-08-09,order,1,3.134375e+12,6825306.0,Ruban adhésif 33 x 19mm SCOTCH,ruban-adhesif-33-x-19mm-scotch,NaN,Matériel de bureau,NaN,2,0.0,0.87,1.74,0.0,1.74,None


In [145]:
df_prod.query("(dateKey == @pd.to_datetime('2024-10-19'))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice,matchedProductLabel
818,2024-10-19,receipt,1,NaN,NaN,125GX16 PAN0% PAN,125GX16 PAN0% PAN,food,NaN,5.5,1,0.000,4.59,4.59,0.00,4.59,None
819,2024-10-19,receipt,1,NaN,NaN,PAIN AUX RAISINS X,PAIN AUX RAISINS X,food,NaN,5.5,1,0.000,2.99,2.99,0.00,2.99,None
820,2024-10-19,receipt,1,NaN,NaN,1L HUILE FLEUR DE,1L HUILE FLEUR DE,food,NaN,5.5,1,0.000,2.69,2.69,0.00,2.69,None
821,2024-10-19,receipt,1,NaN,NaN,FLT 1.5KG ORANGE,FLT 1.5KG ORANGE,food,NaN,5.5,1,0.000,3.99,3.99,0.00,3.99,"FLT 1,5KG ORANGE DESSERT CRF"
822,2024-10-19,receipt,1,NaN,NaN,350G COULOMMIERS,350G COULOMMIERS,food,NaN,5.5,1,0.000,2.95,2.95,0.00,2.95,None
823,2024-10-19,receipt,3,NaN,NaN,BQ TOM CERISE ALL,BQ TOM CERISE ALL,food,NaN,5.5,3,0.000,0.99,2.97,0.00,2.97,BQ 200G CHAMPIGNONS BRUN FQC
824,2024-10-19,receipt,1,NaN,NaN,AUBERGINE VRAC,AUBERGINE VRAC,food,NaN,5.5,1,1.089,1.69,1.84,0.00,1.84,None
825,2024-10-19,receipt,1,NaN,NaN,POMME TERRE STD,POMME TERRE STD,food,NaN,5.5,1,0.754,1.99,1.50,0.00,1.50,None
826,2024-10-19,receipt,1,NaN,NaN,KIWI VERT PIECE,KIWI VERT PIECE,food,NaN,5.5,8,0.000,0.59,4.72,-0.72,4.00,KIWI VERT PIECE
827,2024-10-19,receipt,1,NaN,NaN,KG TOMATE GRAP FR,KG TOMATE GRAP FR,food,NaN,5.5,1,0.874,1.99,1.74,0.00,1.74,None


In [144]:
df_prod.query("(dateKey == @pd.to_datetime('2024-10-07'))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice,matchedProductLabel
866,2024-10-07,order,1,9.713236e+12,7537445.0,Sacs réutilisables consignés,sacs-reutilisables-consignes,NaN,RETURNABLE_BAG,NaN,0,0.0,0.35,0.00,0.00,0.00,None
867,2024-10-07,order,1,3.276552e+12,5296384.0,Kiwi jaune,kiwi-jaune,NaN,Fruits,NaN,6,0.0,0.99,5.94,0.94,6.88,None
868,2024-10-07,order,1,3.560070e+12,3164301.0,Papier toilette Ultra Confort CARREFOUR ESSENTIAL,papier-toilette-ultra-confort-carrefour-essential,NaN,"Essuie-tout, Papier toilette et Mouchoirs",NaN,2,0.0,4.59,9.18,1.38,10.56,None
869,2024-10-07,order,1,3.560071e+12,6557620.0,Œufs frais poules au sol CARREFOUR CLASSIC',oeufs-frais-poules-au-sol-carrefour-classic,NaN,Œufs,NaN,1,0.0,4.55,4.55,0.00,4.55,None


In [132]:
df_prod.query("(dateKey == @pd.to_datetime('2025-02-18'))")

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice,matchedProductLabel
330,2025-02-18,order,1,9.713236e+12,7537445.0,Sacs réutilisables consignés,sacs-reutilisables-consignes,NaN,RETURNABLE_BAG,NaN,2,0.000,0.35,0.70,0.00,0.70,None
331,2025-02-18,order,1,3.560071e+12,4713769.0,Poulet fermier d'Auvergne Label Rouge,poulet-fermier-d-auvergne-label-rouge,NaN,Volaille et Rôtisserie,NaN,1,1.572,10.22,10.22,0.00,10.22,None
332,2025-02-18,order,1,3.564980e+12,5703584.0,Cidre artisanal le brun rosé,cidre-artisanal-le-brun-rose,NaN,Bières et Cidres,NaN,1,0.000,3.89,3.89,0.00,3.89,None
333,2025-02-18,order,1,3.523680e+12,7327283.0,Oignons rouges,oignons-rouges,NaN,Légumes,NaN,1,0.000,0.99,0.99,0.00,0.99,None
334,2025-02-18,order,1,3.499150e+12,7750698.0,Tender de poulet pané épicé,tender-de-poulet-pane-epice,NaN,Viandes,NaN,1,0.000,4.00,4.00,0.00,4.00,None
335,2025-02-18,order,1,8.718952e+12,7710114.0,Liquide Vaisselle Vinaigre et Sel Minéral Bi-A...,liquide-vaisselle-vinaigre-et-sel-mineral-bi-a...,NaN,Nettoyants vaisselle,NaN,1,0.000,4.98,4.98,0.00,4.98,None
336,2025-02-18,order,1,3.428272e+12,2540902.0,Lait Sans lactose UHT Ecrémé Matin Léger,lait-sans-lactose-uht-ecreme-matin-leger,NaN,Laits et Boissons végétales,NaN,2,0.000,11.82,23.64,0.00,23.64,None
337,2025-02-18,order,1,3.523680e+12,6966429.0,Bananes Bio,bananes-bio,NaN,Le Marché,NaN,1,0.000,1.99,1.99,0.00,1.99,None
338,2025-02-18,order,1,3.073781e+12,6955311.0,Fromage Fondu,fromage-fondu,NaN,Fromages,NaN,3,0.000,4.99,14.97,4.99,19.96,None
339,2025-02-18,order,1,3.560070e+12,2828693.0,Mouchoirs Confort,mouchoirs-confort,NaN,"Essuie-tout, Papier toilette et Mouchoirs",NaN,4,0.000,3.65,14.60,2.20,16.80,None


In [84]:
df_loyalty.query("date == @pd.to_datetime('2024-05-10')")

,operationId,date,earned,burned,itemLabel,promotionLabel,itemRd,loyaltyOperation,matchedProductLabel
217,63085345893,2024-05-10,0.26,0.0,1KG FARINE BLE T45 CRF CLASSIC,NaN,0.13,Paiement en caisse,None
218,63085345893,2024-05-10,0.26,0.0,1KG FARINE BLE FLUID T45 CRF C,NaN,0.13,Paiement en caisse,None


In [5]:
display(df_prod)

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice,matchedProductLabel
0,2025-04-29,receipt,2,NaN,NaN,5X10 D FREEDENT ME,5X10 D FREEDENT ME,other,NaN,20.0,2,0.000,2.39,4.78,-1.43,3.35,None
1,2025-04-29,receipt,1,NaN,NaN,KG POULET FERMIER,KG POULET FERMIER,food,NaN,5.5,1,1.912,6.50,12.43,0.00,12.43,FLT 1 KG PDT CONSO 0.99 SIMPL
2,2025-04-29,receipt,1,NaN,NaN,750G OMEGA 3 DX SH,750G OMEGA 3 DX SH,other,NaN,20.0,1,0.000,5.39,5.39,0.00,5.39,750G OMEGA 3 DX ST HUBERT
3,2025-04-29,receipt,1,NaN,NaN,3 RECH RUBAN MAGI,3 RECH RUBAN MAGI,other,NaN,20.0,1,0.000,4.99,4.99,0.00,4.99,MAIS SANS SUCRE AJOUTE BOITE 3
4,2025-04-29,receipt,1,NaN,NaN,132G BISC.COCO SSA,132G BISC.COCO SSA,food,NaN,5.5,1,0.000,2.45,2.45,0.00,2.45,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5014,2022-04-25,receipt,1,NaN,NaN,COURGETTE P10.5 CM,COURGETTE P10.5 CM,other,NaN,10.0,1,0.000,1.75,1.75,0.00,1.75,None
5015,2022-04-25,receipt,1,NaN,NaN,POMME ARIANE,POMME ARIANE,food,NaN,5.5,1,0.000,2.50,2.50,0.00,2.50,"ST 1,5KG POMME GALA FR"
5016,2022-04-25,receipt,1,NaN,NaN,PIECE ANANAS EXTRA,PIECE ANANAS EXTRA,food,NaN,5.5,1,0.000,1.99,1.99,0.00,1.99,AVOCAT PIECE
5017,2022-04-25,receipt,1,NaN,NaN,MENTHE FRAI,MENTHE FRAI,food,NaN,5.5,1,0.000,0.61,0.61,0.00,0.61,MENTHE FRAICHE BOTTE
